In [4]:
# ============================================================
# Cell 1: load data
# ============================================================
import re
import pandas as pd
import os
from collections import defaultdict
from difflib import SequenceMatcher

notebook_path = os.getcwd()
in_dir_notifications_raw = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "notification_historical", "notification.jsonl"))
in_dir_master_direction   = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "master_directory", "master_directory.jsonl"))

circulars         = pd.read_json(in_dir_notifications_raw, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines=True)
print(f"circulars: {len(circulars)}, master_directions: {len(master_directions)}")


# ============================================================
# Cell 2: citation regex + normalizer
# ============================================================
DOC_TYPE = r"(?:Directions?|Guidelines?|Regulations?|Rules?|Circulars?|Framework|Scheme)"

p1 = re.compile(
    r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)"
    r"(?:\s*\([^)]*\))*"
    r"\s*(?:[A-Za-z]+\s+){0,3}" + DOC_TYPE,
    re.IGNORECASE
)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)

norm = lambda s: re.sub(r"\s+", " ", s.replace("–", "-").replace("—", "-")).strip(" ,-").lower()

def extract_names(text):
    if not isinstance(text, str):
        return []
    return [norm(x) for x in (p1.findall(text) + p2.findall(text))]


# ============================================================
# Cell 3: title + full-text + "lead paragraph" extraction
# ============================================================
# Lead cutoff (start of numbered paragraph "2.") is ONLY for the "does this
# doc even carry a citation" bucketing check further down. Matching itself
# always runs on found_text (full text) -- restricting matching to the lead
# was a real bug that cut real matches from 102 to 79.
lead_break = re.compile(r"\n\s*2\.\s")

def get_lead(text):
    if not isinstance(text, str):
        return ""
    m = lead_break.search(text)
    return text[:m.start()] if m else text[:600]

circulars["found_title"]     = circulars["title"].apply(extract_names)                        # title citations
circulars["found_text"]      = circulars["text"].apply(extract_names)                         # full text -- used for matching
circulars["found_text_lead"] = circulars["text"].apply(lambda t: extract_names(get_lead(t)))  # lead only -- used for bucketing


# ============================================================
# Cell 4: build master_lookup, flagging collisions instead of silently overwriting
# ============================================================
master_directions["extracted_names"] = master_directions["title"].apply(extract_names)

name_to_ids = defaultdict(set)
for _, r in master_directions.iterrows():
    for name in r["extracted_names"]:
        name_to_ids[name].add(r["id"])

collisions = {k: v for k, v in name_to_ids.items() if len(v) > 1}
print(f"{len(collisions)} names collide across master_directions (excluded from lookup):")
print(collisions)

unlinkable = master_directions[master_directions["extracted_names"].str.len() == 0]
print(f"{len(unlinkable)}/{len(master_directions)} master directions have no extractable name (can never be matched to)")

master_lookup = {name: next(iter(ids)) for name, ids in name_to_ids.items() if len(ids) == 1}
master_keys = list(master_lookup.keys())


# ============================================================
# Cell 5: NBFC relevance + discard signal -- computed BEFORE matching, so
# the title/text lookup against master_lookup is only attempted on
# circulars that could plausibly be NBFC-relevant.
#
# Two separate NBFC signals, used for two separate jobs:
#   - is_nbfc_in_title : TITLE ONLY. Used solely to decide whether a
#     circular that names a different specific entity in its title should
#     still be spared from discard. A body-text-only NBFC mention (e.g. a
#     comparative aside) is not a scope claim and must not block discard --
#     confirmed by 5 real cases (ids 58, 155, 172, 202, 234) that were
#     misrouted into nbfc_citation_unmatched before this split existed.
#   - is_nbfc_relevant : TITLE OR TEXT. Kept for bucket()'s separate job
#     downstream -- flagging genuinely NBFC-relevant no_match rows that
#     didn't name any other specific entity, where a body-text mention is
#     a legitimate (if weaker) signal once discard is already off the table.
# ============================================================
nbfc_pattern = re.compile(
    r"non[\s-]?banking financial compan|nbfc"
    r"|core investment compan(?:y|ies)"          # dropped bare \bcic\b — collides with Credit Information Company
    r"|standalone primary dealer|\bspd\b"
    r"|mortgage guarantee compan(?:y|ies)|\bmgc\b"
    r"|non-?operative financial holding compan(?:y|ies)|\bnofhc\b"
    r"|housing finance compan(?:y|ies)|\bhfc\b",
    re.IGNORECASE
)
circulars["is_nbfc_relevant"] = (
    circulars["text"].str.contains(nbfc_pattern, na=False) |
    circulars["title"].str.contains(nbfc_pattern, na=False)
)

other_entity_pattern = re.compile(
    r"regional rural bank"
    r"|urban co-?operative bank"
    r"|rural co-?operative bank"
    r"|state co-?operative bank"
    r"|district central co-?operative bank"
    r"|scheduled commercial bank"
    r"|commercial bank"
    r"|payments? bank"
    r"|small finance bank"
    r"|local area bank"
    r"|co-?operative bank"
    r"|banker and debt manager to government"
    r"|banker to governments? and banks"
    r"|consumer education and protection"
    r"|all india financial institutions?"
    r"|asset reconstruction compan(?:y|ies)"
    r"|credit information compan(?:y|ies)"
    r"|financial inclusion and development"
    r"|financial market"
    r"|issuer of currency"
    r"|payments? and settlement systems?",
    re.IGNORECASE
)
circulars["is_nbfc_in_title"] = circulars["title"].str.contains(nbfc_pattern, na=False)
circulars["is_nbfc_relevant"] = (
    circulars["text"].str.contains(nbfc_pattern, na=False) |
    circulars["is_nbfc_in_title"]
)

...

circulars["names_other_entity"] = circulars["title"].str.contains(other_entity_pattern, na=False)

circulars["discard_pre_match"] = circulars["names_other_entity"] & ~circulars["is_nbfc_relevant"]

print(f"{circulars['discard_pre_match'].sum()} / {len(circulars)} circulars discarded "
      f"before matching (named a different entity, not NBFC)")



# ============================================================
# Cell 6 (was Cell 5): title/full-text matching -- skips anything already
# discarded, so those rows never get a chance at a spurious title/text match.
# ============================================================
def match_row(row):
    if row["discard_pre_match"]:
        return None, "discarded"
    for n in row["found_title"]:
        if n in master_lookup:
            return master_lookup[n], "title_match"
    for n in row["found_text"]:          # full text, not lead
        if n in master_lookup:
            return master_lookup[n], "text_match"
    return None, "no_match"

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(match_row(r)), axis=1)
print(circulars["match_method"].value_counts())

recovered_by_full_text = circulars[
    (circulars["match_method"].isin(["title_match", "text_match"])) &   # excludes "discarded" too
    (circulars["found_title"].str.len() == 0) &
    (~circulars["found_text_lead"].apply(lambda lst: any(n in master_lookup for n in lst)))
]
print(f"{len(recovered_by_full_text)} matches only found past the lead cutoff")
recovered_by_full_text[["id", "title", "matched_id"]]


# ============================================================
# Cell 7 (was Cell 6): fuzzy candidates -- unchanged. Naturally only runs
# on match_method == "no_match" rows, so discarded rows are excluded here
# for free (they're labeled "discarded", not "no_match").
# ============================================================
def char_diff(a, b):
    diff = 0
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b).get_opcodes():
        if tag != "equal":
            diff += max(i2 - i1, j2 - j1)
    return diff

fuzzy_candidates = []
for idx, row in circulars[circulars["match_method"] == "no_match"].iterrows():
    for n in (row["found_title"] + row["found_text_lead"]):
        for key in master_keys:
            if char_diff(n, key) <= 5:
                fuzzy_candidates.append((idx, row["id"], n, key))
                break

fuzzy_df = pd.DataFrame(fuzzy_candidates, columns=["row_idx", "circular_id", "extracted_name", "closest_master_name"])
print(f"{len(fuzzy_df)} candidates")
fuzzy_df


# ============================================================
# Cell 7b (was Cell 6b): apply fuzzy matches -- unchanged.
# ============================================================
dupe_check = fuzzy_df.groupby("row_idx")["closest_master_name"].apply(
    lambda names: len({master_lookup[n] for n in names})
)
ambiguous_rows = dupe_check[dupe_check > 1].index
if len(ambiguous_rows):
    print(f"{len(ambiguous_rows)} rows have conflicting fuzzy candidates -- review before applying:")
    print(fuzzy_df[fuzzy_df["row_idx"].isin(ambiguous_rows)])

safe_fuzzy = fuzzy_df[~fuzzy_df["row_idx"].isin(ambiguous_rows)].drop_duplicates("row_idx")
for _, r in safe_fuzzy.iterrows():
    circulars.loc[r["row_idx"], "matched_id"]   = master_lookup[r["closest_master_name"]]
    circulars.loc[r["row_idx"], "match_method"] = "fuzzy_match"

print(circulars["match_method"].value_counts())


# ============================================================
# Cell 8 (was Cell 7, simplified): bucket assignment. discard is already
# decided pre-match, so this just reads match_method off directly instead
# of re-deriving it from is_nbfc_relevant / names_other_entity.
# ============================================================
def bucket(row):
    if row["match_method"] == "discarded":
        return "discard"
    if row["match_method"] != "no_match":
        return "linked"
    if row["is_nbfc_relevant"]:
        has_citation = len(row["found_title"]) > 0 or len(row["found_text_lead"]) > 0
        return "nbfc_citation_unmatched" if has_citation else "nbfc_standalone"
    return "general"   # not discarded, not NBFC-named, not matched -- genuinely ambiguous

circulars["bucket"] = circulars.apply(bucket, axis=1)
print(circulars["bucket"].value_counts())


# ============================================================
# Cell 9 (was Cell 8): subject_code cross-check -- filter tightened to the
# real match methods, so a "discarded" row (matched_id=None) never gets
# compared against a coincidental subject_code hit and flagged as a
# disagreement.
# ============================================================
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")
circulars["subject_code"]         = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

code_counts = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
generic_codes = code_counts[code_counts > 1].index.tolist()
clean_master = master_directions[~master_directions["subject_code"].isin(generic_codes)]

code_matches = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)
both = code_matches[
    code_matches["match_method"].isin(["title_match", "text_match", "fuzzy_match"]) &
    code_matches["id_master"].notna()
]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between regex-match and code-match")


# ============================================================
# Cell 10 (was Cell 9): final tally -- unchanged logic.
# ============================================================
print(circulars["bucket"].value_counts())
nbfc_total = circulars["bucket"].isin(["linked", "nbfc_citation_unmatched", "nbfc_standalone"]).sum()
print(f"NBFC-relevant: {nbfc_total} / {len(circulars)}")
print(f"Discard (named a different specific entity): {(circulars['bucket']=='discard').sum()} / {len(circulars)}")
print(f"General (ambiguous, needs model review): {(circulars['bucket']=='general').sum()} / {len(circulars)}")

circulars: 766, master_directions: 44
1 names collide across master_directions (excluded from lookup):
{'non-banking financial companies - miscellaneous': {13586, 12931}}
0/44 master directions have no extractable name (can never be matched to)
445 / 766 circulars discarded before matching (named a different entity, not NBFC)
match_method
discarded      445
no_match       220
title_match     76
text_match      25
Name: count, dtype: int64
1 matches only found past the lead cutoff
3 candidates
match_method
discarded      445
no_match       218
title_match     76
text_match      25
fuzzy_match      2
Name: count, dtype: int64
bucket
discard                    445
nbfc_citation_unmatched    138
linked                     103
general                     79
nbfc_standalone              1
Name: count, dtype: int64
0 / 7 disagree between regex-match and code-match
bucket
discard                    445
nbfc_citation_unmatched    138
linked                     103
general                     79

In [5]:
pd.set_option('display.max_colwidth', None)
def review_sample(method, n=15, seed=0):
    pool = circulars[circulars["match_method"] == method]
    sample = pool.sample(min(n, len(pool)), random_state=seed).copy()
    return sample.merge(
        master_directions[["id", "title"]].rename(columns={"title": "master_title"}),
        left_on="matched_id", right_on="id", suffixes=("", "_m")
    )[["id", "title", "master_title", "match_method"]]

review_sample("title_match")
review_sample("text_match")
review_sample("fuzzy_match")   # review ALL of these, not just a sample, if the count is small

,id,title,master_title,match_method
0,13379,"Reserve Bank of India (Non-Banking Financial Companies– Undertaking of Financial Services) –Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Undertaking of Financial Services) Directions, 2025 (Updated as on July 01, 2026)",fuzzy_match
1,13213,"Reserve Bank of India (Non-Operative Financial Holding Company) (Amendment) Directions, 2025","Reserve Bank of India (Non-Operative Financial Holding Companies) Directions, 2025 (Updated as on December 05, 2025)",fuzzy_match


In [6]:
code_lookup = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code", "title"]],
    on="subject_code", how="left", suffixes=("", "_code_master")
)

recoverable = code_lookup[
    (code_lookup["match_method"] == "no_match") &
    code_lookup["id_code_master"].notna()
]
print(f"{len(recoverable)} unmatched circulars have a subject_code hit against a master direction")
recoverable[["id", "title", "title_code_master"]]

5 unmatched circulars have a subject_code hit against a master direction


,id,title,title_code_master
58,12983,"Reserve Bank of India (All India Financial Institutions - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on June 16, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
155,13080,"Reserve Bank of India (Local Area Banks – Prudential Norms on Capital Adequacy) Directions, 2025","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
172,13097,"Reserve Bank of India (Payments Banks – Prudential Norms on Capital Adequacy) Directions, 2025 (updated as on May 08, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
202,13127,"Reserve Bank of India (Small Finance Banks – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
234,13159,"Reserve Bank of India (Commercial Banks - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"


In [7]:
matched_ids = set(circulars.loc[circulars["match_method"] != "no_match", "matched_id"])
orphans = master_directions[~master_directions["id"].isin(matched_ids)]
nbfc_orphans = orphans[orphans["title"].str.contains(nbfc_pattern, na=False)]
print(f"{len(orphans)}/{len(master_directions)} master directions have zero linked circulars")
print(f"{len(nbfc_orphans)} of those are NBFC-titled")
nbfc_orphans[["id", "title"]]

2/44 master directions have zero linked circulars
2 of those are NBFC-titled


,id,title
0,12931,"Reserve Bank of India (Non-Banking Financial Companies – Miscellaneous) Directions, 2025 (Updated as on February 26, 2026)"
36,13586,"Reserve Bank of India (Non-Banking Financial Companies – Miscellaneous) Supervisory Directions, 2026"


In [8]:
# is_nbfc_relevant checks text too, so nothing in discard should contain the keyword
assert circulars.loc[circulars["bucket"] == "discard", "is_nbfc_relevant"].sum() == 0

# "general" should mean neither pattern hit - confirm no overlap
general_check = circulars[circulars["bucket"] == "general"]
assert general_check["is_nbfc_relevant"].sum() == 0
assert general_check["names_other_entity"].sum() == 0

In [9]:
# fill in ~15-20 pairs you've manually verified, including the Concentration
# Risk case you already confirmed
gold_pairs = {
    "circular_id_1": "master_id_1",
    "circular_id_2": "master_id_2",
}

for cid, expected in gold_pairs.items():
    row = circulars.loc[circulars["id"] == cid, "matched_id"]
    got = row.values[0] if len(row) else None
    status = "OK" if got == expected else f"MISMATCH (expected {expected}, got {got})"
    print(cid, status)

circular_id_1 MISMATCH (expected master_id_1, got None)
circular_id_2 MISMATCH (expected master_id_2, got None)


In [10]:
circulars.info()

<class 'pandas.DataFrame'>
RangeIndex: 766 entries, 0 to 765
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  766 non-null    int64  
 1   title               766 non-null    str    
 2   category            766 non-null    str    
 3   url                 766 non-null    str    
 4   text                766 non-null    str    
 5   found_title         766 non-null    object 
 6   found_text          766 non-null    object 
 7   found_text_lead     766 non-null    object 
 8   is_nbfc_relevant    766 non-null    bool   
 9   is_nbfc_in_title    766 non-null    bool   
 10  names_other_entity  766 non-null    bool   
 11  discard_pre_match   766 non-null    bool   
 12  matched_id          103 non-null    float64
 13  match_method        766 non-null    str    
 14  bucket              766 non-null    str    
 15  subject_code        379 non-null    str    
dtypes: bool(4), float64

In [11]:
pd.set_option('display.max_colwidth', 50)
print(circulars.iloc(1))

In [12]:
pd.set_option('display.max_colwidth', 500)
print(circulars.iloc[1])

id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  12926
title                                                                                                                                                                                                                                                                                                                                                                                                                                 Reserve Bank of India (Credit Information Companies) Dir

In [13]:
import os
notebook_path = os.getcwd()
out_path = os.path.join(notebook_path,"..","..","Data","notification_historical", "circulars_filtered.jsonl")

kept = circulars[circulars["bucket"] != "discard"].copy()

kept.to_json(out_path, orient="records", lines=True, force_ascii=False)

print(f"Saved {len(kept)} / {len(circulars)} circulars to {out_path}")
print(kept["bucket"].value_counts())

Saved 321 / 766 circulars to C:\Users\VEDANG BARMAN\Desktop\Regulatory Compliance Agent\sandbox\jupyter\..\..\Data\notification_historical\circulars_filtered.jsonl
bucket
nbfc_citation_unmatched    138
linked                     103
general                     79
nbfc_standalone              1
Name: count, dtype: int64
